In [ ]:
# =============================================================================
# IMPORTS
# =============================================================================
import os
import platform
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.utils import resample
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report, roc_auc_score, roc_curve, precision_recall_curve, auc
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
import hashlib
import random
import time
import joblib

# Sklearn classifiers for Apollon
from sklearn.ensemble import RandomForestClassifier as skRF
from sklearn.tree import DecisionTreeClassifier as skDT
from sklearn.naive_bayes import GaussianNB as skNB
from sklearn.linear_model import LogisticRegression as skLR
from sklearn.neural_network import MLPClassifier as skMLP
from sklearn.cluster import KMeans as skKMeans

# GPU acceleration for Apollon (cuML/RAPIDS)
USE_CUML = False
try:
    import cupy as cp
    import cudf
    from cuml.ensemble import RandomForestClassifier as cuRF
    from cuml.tree import DecisionTreeClassifier as cuDT
    from cuml.naive_bayes import GaussianNB as cuNB
    from cuml.linear_model import LogisticRegression as cuLR
    from cuml.cluster import KMeans as cuKMeans
    USE_CUML = True
    print("\n✅ cuML (RAPIDS) available for Apollon GPU acceleration")
except ImportError:
    print("\n⚠️ cuML not available - Apollon will use CPU sklearn")

# Set Apollon classifier classes based on GPU availability
if USE_CUML:
    RandomForestClassifier = cuRF
    DecisionTreeClassifier = cuDT
    GaussianNB = cuNB
    LogisticRegression = cuLR
    KMeansCluster = cuKMeans
    print("🚀 Apollon using GPU-accelerated cuML classifiers")
else:
    RandomForestClassifier = skRF
    DecisionTreeClassifier = skDT
    GaussianNB = skNB
    LogisticRegression = skLR
    KMeansCluster = skKMeans
    print("🔧 Apollon using CPU sklearn classifiers")

# =============================================================================
# TPU/GPU DETECTION
# =============================================================================
print("\n" + "="*80)
print("HARDWARE DETECTION")
print("="*80)
print(f"PyTorch: {torch.__version__}")
print(f"Platform: {platform.platform()}")

# TPU setup
TPU_AVAILABLE = False
xm = None
xmp = None

try:
    import torch_xla
    import torch_xla.core.xla_model as xla_model
    TPU_AVAILABLE = True
    xm = xla_model
    print(f"✅ TPU Available: torch_xla {torch_xla.__version__}")
    try:
        import torch_xla.distributed.xla_multiprocessing as xla_mp
        xmp = xla_mp
    except ImportError:
        pass
except ImportError:
    print("⚠️ TPU not available (torch_xla not installed)")
    print("   For Kaggle TPU: Runtime → Change runtime type → TPU")

# CUDA setup
if torch.cuda.is_available():
    print(f"\n✅ CUDA Available")
    print(f"   GPU Count: {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f"   GPU {i}: {props.name} ({props.total_memory / 1e9:.2f} GB)")
else:
    print("\n⚠️ CUDA not available")

print("="*80 + "\n")

# Device selection
USE_XLA = False
DEVICE = None

if TPU_AVAILABLE and xm is not None:
    USE_XLA = True
    # Use torch_xla.device() instead of deprecated xm.xla_device()
    try:
        import torch_xla
        DEVICE = torch_xla.device()
    except:
        # Fallback to old method if new method fails
        DEVICE = xm.xla_device()
    print(f"🎯 Using TPU: {DEVICE}")
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    print(f"🎯 Using CUDA: {DEVICE}")
else:
    DEVICE = torch.device("cpu")
    print(f"🎯 Using CPU: {DEVICE}")

print("")



# =============================================================================
# CONFIGURATION
# =============================================================================

# Dataset paths - CIC-DDoS2019
TRAIN_DATA_DIR = "/kaggle/input/cic-ddos2019-30gb-full-dataset-csv-files/01-12"
TEST_DATA_DIR = "/kaggle/input/cic-ddos2019-30gb-full-dataset-csv-files/03-11"

# Training files (Day 1 - 01-12)
TRAIN_FILES = [
    "TFTP.csv",
    "DrDoS_LDAP.csv",
    "DrDoS_MSSQL.csv",
    "DrDoS_NetBIOS.csv",
    "DrDoS_NTP.csv",
    "DrDoS_SNMP.csv",
    "DrDoS_SSDP.csv",
    "DrDoS_UDP.csv",
    "Syn.csv",
    "DrDoS_DNS.csv",
    "UDPLag.csv"
]

# Test files (Day 2 - 03-11)
TEST_FILES = [
    "LDAP.csv",
    "MSSQL.csv",
    "NetBIOS.csv",
    "Portmap.csv",
    "Syn.csv"
    # Note: UDP.csv and UDPLag.csv can be added if fixed
]

# Downsampling configuration
MULT = 5  # Multiplier to control anomaly class size (attack flows = benign flows * MULT)

# Toggle between test mode and full data mode
USE_FULL_DATA = True  # Set to True for full dataset, False for quick testing
NUM_SAMPLES = None if USE_FULL_DATA else 5000  # None = unlimited, 5000 = quick test

# Output
OUT_DIR = "/kaggle/working/models"
PLOT_DIR = "/kaggle/working/plots"
PROCESSED_DATA_DIR = "/kaggle/working"
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(PLOT_DIR, exist_ok=True)

# Set seed for reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
random.seed(SEED)


# =============================================================================
# DATA LOADING AND PREPROCESSING FUNCTIONS
# =============================================================================

def string2numeric_hash(text):
    """Convert string to numeric hash for timestamp processing."""
    return int(hashlib.md5(text.encode()).hexdigest()[:8], 16)


def load_file(path, num_samples=NUM_SAMPLES, mult=MULT):
    """
    Load and downsample a single CSV file.
    
    Balances benign and anomalous flows by downsampling the majority class.
    Anomalous flows can be 'mult' times larger than benign flows to prevent
    excessive information loss while reducing class imbalance.
    
    Args:
        path: Path to CSV file
        num_samples: Maximum number of rows to read (None for unlimited)
        mult: Multiplier for target anomaly class size
        
    Returns:
        Tuple of (flows_ok, flows_ddos_reduced): Two separate DataFrames for benign and attack flows
    """
    # Count total lines in the file (excluding header)
    total_lines = sum(1 for _ in open(path, encoding='utf-8', errors='ignore')) - 1
    
    # Ensure at most num_samples rows are read randomly
    if num_samples and total_lines > num_samples:
        skip_rows = sorted(random.sample(range(1, total_lines + 1), total_lines - num_samples))
    else:
        skip_rows = None
    
    # Read CSV file
    data = pd.read_csv(path, skiprows=skip_rows, sep=',', low_memory=False, encoding='utf-8', encoding_errors='ignore')
    
    # Separate benign and anomalous flows
    is_benign = data[' Label'] == 'BENIGN'
    flows_ok = data[is_benign]
    flows_ddos_full = data[~is_benign]
    
    # Calculate target size for anomalous data
    sizeDownSample = len(flows_ok) * mult
    
    # Downsample majority class if necessary
    if sizeDownSample < len(flows_ddos_full):
        flows_ddos_reduced = resample(
            flows_ddos_full,
            replace=False,
            n_samples=sizeDownSample,
            random_state=SEED
        )
    else:
        flows_ddos_reduced = flows_ddos_full
    
    # Return two separate DataFrames (DO NOT concatenate here)
    return flows_ok, flows_ddos_reduced


def load_huge_file(path, num_samples=NUM_SAMPLES, mult=MULT, chunksize=500000):
    """
    Load and downsample a large CSV file in chunks.
    
    For very large files, reads and processes data in chunks to manage memory.
    Each chunk is processed separately, then combined at the end.
    
    Args:
        path: Path to CSV file
        num_samples: Maximum number of rows to read (None for unlimited)
        mult: Multiplier for target anomaly class size
        chunksize: Number of rows per chunk
        
    Returns:
        Tuple of (flows_ok, flows_ddos): Two separate DataFrames for benign and attack flows
    """
    # Count total lines
    total_lines = sum(1 for _ in open(path, encoding='utf-8', errors='ignore')) - 1
    
    # Determine which rows to skip
    if num_samples and total_lines > num_samples:
        skip_rows = sorted(random.sample(range(1, total_lines + 1), total_lines - num_samples))
    else:
        skip_rows = None
    
    # Read in chunks
    df_chunk = pd.read_csv(path, skiprows=skip_rows, chunksize=chunksize, low_memory=False, encoding='utf-8', encoding_errors='ignore')
    
    chunk_list_ok = []
    chunk_list_ddos = []
    
    # Process each chunk
    for chunk in df_chunk:
        is_benign = chunk[' Label'] == 'BENIGN'
        flows_ok = chunk[is_benign]
        flows_ddos_full = chunk[~is_benign]
        
        # Downsample anomalous flows in this chunk
        if (len(flows_ok) * mult) < len(flows_ddos_full):
            sizeDownSample = len(flows_ok) * mult
            flows_ddos_reduced = resample(
                flows_ddos_full,
                replace=False,
                n_samples=sizeDownSample,
                random_state=SEED
            )
        else:
            flows_ddos_reduced = flows_ddos_full
        
        chunk_list_ok.append(flows_ok)
        chunk_list_ddos.append(flows_ddos_reduced)
    
    # Combine all chunks but keep benign and attack separate
    flows_ok = pd.concat(chunk_list_ok)
    flows_ddos = pd.concat(chunk_list_ddos)
    
    # Return two separate DataFrames (DO NOT concatenate here)
    return flows_ok, flows_ddos


def preprocess_dataframe(df, dataset_type='train'):
    """
    Preprocess CIC-DDoS2019 dataset.
    
    Steps:
    1. Replace infinite values with 0
    2. Convert numerical columns safely
    3. Map attack labels to binary (0=BENIGN, 1=ATTACK)
    4. Hash timestamps to numeric values (preserves temporal information)
    5. Drop unnecessary columns
    
    Args:
        df: Raw DataFrame
        dataset_type: 'train' or 'test' (affects label mapping)
        
    Returns:
        Preprocessed DataFrame
    """
    # Replace infinite values
    df = df.replace(['Infinity', np.inf, -np.inf], 0)
    
    # Convert numerical columns safely
    if ' Flow Packets/s' in df.columns:
        df[' Flow Packets/s'] = pd.to_numeric(df[' Flow Packets/s'], errors='coerce').fillna(0)
    if 'Flow Bytes/s' in df.columns:
        df['Flow Bytes/s'] = pd.to_numeric(df['Flow Bytes/s'], errors='coerce').fillna(0)
    
    # Map labels to binary classification
    if dataset_type == 'train':
        # Day 1 attack types
        label_map = {
            'BENIGN': 0,
            'DrDoS_DNS': 1,
            'DrDoS_LDAP': 1,
            'DrDoS_MSSQL': 1,
            'DrDoS_NTP': 1,
            'DrDoS_NetBIOS': 1,
            'DrDoS_SNMP': 1,
            'DrDoS_SSDP': 1,
            'DrDoS_UDP': 1,
            'Syn': 1,
            'TFTP': 1,
            'UDP-lag': 1,
            'WebDDoS': 1
        }
    else:  # test
        # Day 2 attack types
        label_map = {
            'BENIGN': 0,
            'LDAP': 1,
            'NetBIOS': 1,
            'MSSQL': 1,
            'Portmap': 1,
            'Syn': 1
        }
    
    df[' Label'] = df[' Label'].replace(label_map).astype(int)
    
    # Hash Timestamp to numeric value (like compare.md)
    # Format: "DD/MM/YYYY HH:MM:SS.microseconds" → extract time → hash
    if ' Timestamp' in df.columns:
        # Split timestamp: "DD/MM/YYYY HH:MM:SS" → keep only time part
        colunaTime = df[' Timestamp'].str.split(' ', n=1, expand=True)
        if colunaTime.shape[1] == 2:
            # Get time part (HH:MM:SS.microseconds)
            time_part = colunaTime[1].str.split('.', n=1, expand=True)[0]  # Remove microseconds
            # Hash the time string to numeric
            df[' Timestamp'] = time_part.apply(lambda x: string2numeric_hash(str(x)) if pd.notna(x) else 0)
        else:
            # Fallback: if format is different, just hash the whole thing
            df[' Timestamp'] = df[' Timestamp'].apply(lambda x: string2numeric_hash(str(x)) if pd.notna(x) else 0)
    
    # Drop unnecessary columns (but keep hashed Timestamp)
    columns_to_drop = []
    for col in [' Source IP', ' Destination IP', 'Flow ID', 'SimillarHTTP', 'Unnamed: 0']:
        if col in df.columns:
            columns_to_drop.append(col)
    
    if columns_to_drop:
        df.drop(columns=columns_to_drop, inplace=True)
    
    return df


def normalize_data(X_train, X_test):
    """
    Normalize data between -1 and 1 using MinMaxScaler.
    
    Args:
        X_train: Training features
        X_test: Test features
        
    Returns:
        Normalized X_train, X_test
    """
    scaler = MinMaxScaler(feature_range=(-1, 1)).fit(X_train)
    X_train = scaler.transform(X_train)
    X_test = scaler.transform(X_test)
    return X_train, X_test


def train_test_split_custom(samples, test_size=0.33):
    """
    Split samples into train and test sets.
    
    Args:
        samples: DataFrame with features and label
        test_size: Fraction of data for testing
        
    Returns:
        X_train, X_test, y_train, y_test
    """
    X = samples.iloc[:, :-1]  # All columns except last
    y = samples.iloc[:, -1]   # Last column (label)
    
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=SEED
    )
    
    return X_train, X_test, y_train, y_test


def test_normal_atk(y_test, y_pred):
    """
    Calculate detection rates for normal and attack flows separately.
    
    Args:
        y_test: True labels
        y_pred: Predicted labels
        
    Returns:
        normal_detect_rate, atk_detect_rate
    """
    df = pd.DataFrame({'y_test': y_test, 'y_pred': y_pred})
    
    normal = df[df['y_test'] == 0].shape[0]
    atk = df[df['y_test'] == 1].shape[0]
    
    wrong = df[df['y_test'] != df['y_pred']]
    wrong_counts = wrong.groupby('y_test').count()
    
    wrong_normal = wrong_counts.loc[0]['y_pred'] if 0 in wrong_counts.index else 0
    wrong_atk = wrong_counts.loc[1]['y_pred'] if 1 in wrong_counts.index else 0
    
    normal_detect_rate = (normal - wrong_normal) / normal if normal > 0 else 0
    atk_detect_rate = (atk - wrong_atk) / atk if atk > 0 else 0
    
    return normal_detect_rate, atk_detect_rate


def format_3d(df):
    """
    Reshape data in 3D format for RNN/LSTM/GRU inputs.
    
    Args:
        df: Input DataFrame or array
        
    Returns:
        Reshaped array with shape (samples, timesteps, features)
    """
    X = np.array(df)
    return np.reshape(X, (X.shape[0], X.shape[1], 1))


def format_2d(df):
    """
    Reshape data in 2D format for standard ML inputs.
    
    Args:
        df: Input DataFrame or array
        
    Returns:
        Reshaped array with shape (samples, features)
    """
    X = np.array(df)
    return np.reshape(X, (X.shape[0], X.shape[1]))


def compile_train(model, X_train, y_train, deep=True, epochs=10, batch_size=256, 
                  learning_rate=0.001, verbose=1, plot_history=True):
    """
    Unified function to compile and train learning models.
    
    Args:
        model: PyTorch model or scikit-learn model
        X_train: Training features
        y_train: Training labels
        deep: If True, assumes PyTorch deep learning model. If False, assumes scikit-learn
        epochs: Number of training epochs (for deep learning only)
        batch_size: Batch size (for deep learning only)
        learning_rate: Learning rate (for deep learning only)
        verbose: Verbosity level
        plot_history: Whether to plot training history (for deep learning only)
        
    Returns:
        Trained model (and history dict for deep learning models)
    """
    if deep:
        # PyTorch deep learning training
        model = model.to(DEVICE)
        criterion = nn.CrossEntropyLoss()
        optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
        
        # Convert to tensors if not already
        if not isinstance(X_train, torch.Tensor):
            X_train = torch.FloatTensor(X_train)
            y_train = torch.LongTensor(y_train)
        
        # Create DataLoader
        train_dataset = TensorDataset(X_train, y_train)
        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
        
        history = {'loss': [], 'accuracy': []}
        
        print(f"Training PyTorch model for {epochs} epochs...")
        model.train()
        
        for epoch in range(epochs):
            epoch_loss = 0.0
            correct = 0
            total = 0
            
            for inputs, labels in train_loader:
                inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
                
                optimizer.zero_grad()
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                loss.backward()
                optimizer.step()
                
                epoch_loss += loss.item()
                _, predicted = outputs.max(1)
                total += labels.size(0)
                correct += predicted.eq(labels).sum().item()
            
            epoch_loss /= len(train_loader)
            epoch_acc = correct / total
            
            history['loss'].append(epoch_loss)
            history['accuracy'].append(epoch_acc)
            
            if verbose:
                print(f"Epoch {epoch+1}/{epochs} - Loss: {epoch_loss:.4f}, Accuracy: {epoch_acc*100:.2f}%")
        
        # Plot training history
        if plot_history:
            fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
            
            # Accuracy plot
            ax1.plot(history['accuracy'])
            ax1.set_title('Model Accuracy')
            ax1.set_ylabel('Accuracy')
            ax1.set_xlabel('Epoch')
            ax1.legend(['Train'], loc='upper left')
            ax1.grid(alpha=0.3)
            
            # Loss plot
            ax2.plot(history['loss'])
            ax2.set_title('Model Loss')
            ax2.set_ylabel('Loss')
            ax2.set_xlabel('Epoch')
            ax2.legend(['Train'], loc='upper left')
            ax2.grid(alpha=0.3)
            
            plt.tight_layout()
            plt.savefig(os.path.join(PLOT_DIR, 'training_history.png'), dpi=150)
            plt.close()
        
        print('✅ PyTorch Model Compiled and Trained')
        return model, history
    
    else:
        # Scikit-learn training (SVM, Random Forest, etc.)
        print("Training scikit-learn model...")
        model.fit(X_train, y_train)
        print('✅ Scikit-learn Model Trained')
        return model


def testes(model, X_test, y_test, y_pred=None, deep=True, verbose=1):
    """
    Comprehensive testing function with multiple performance metrics.
    
    Args:
        model: Trained model (PyTorch or scikit-learn)
        X_test: Test features
        y_test: True test labels
        y_pred: Predicted labels (optional, will be generated if None)
        deep: If True, assumes PyTorch model. If False, assumes scikit-learn
        verbose: Verbosity level
        
    Returns:
        Tuple of (accuracy, precision, recall, f1, average)
    """
    # Generate predictions if not provided
    if y_pred is None:
        if deep:
            model = model.to(DEVICE)
            model.eval()
            
            if not isinstance(X_test, torch.Tensor):
                X_test = torch.FloatTensor(X_test)
            
            with torch.no_grad():
                X_test = X_test.to(DEVICE)
                outputs = model(X_test)
                
                # Evaluate model (calculate loss if possible)
                if isinstance(y_test, torch.Tensor):
                    y_test_tensor = y_test.to(DEVICE)
                    criterion = nn.CrossEntropyLoss()
                    loss = criterion(outputs, y_test_tensor)
                    if verbose:
                        print(f"Test Loss: {loss.item():.4f}")
                
                _, y_pred = outputs.max(1)
                y_pred = y_pred.cpu().numpy()
        else:
            # Scikit-learn prediction
            y_pred = model.predict(X_test)
    
    # Ensure y_test is numpy array
    if isinstance(y_test, torch.Tensor):
        y_test = y_test.cpu().numpy()
    
    # Calculate metrics
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    
    # Calculate average
    avrg = (acc + prec + rec + f1) / 4
    
    if verbose:
        print('\n' + '='*80)
        print('TEST RESULTS')
        print('='*80)
        print(f'Accuracy:  {acc*100:>6.2f}%')
        print(f'Precision: {prec*100:>6.2f}%')
        print(f'Recall:    {rec*100:>6.2f}%')
        print(f'F1 Score:  {f1*100:>6.2f}%')
        print(f'Average:   {avrg*100:>6.2f}%')
        print('='*80)
        
        # Confusion matrix
        cm = confusion_matrix(y_test, y_pred)
        print('\nConfusion Matrix:')
        print(f"           Predicted")
        print(f"           Benign  Attack")
        print(f"Actual Benign  {cm[0,0]:>6,}  {cm[0,1]:>6,}")
        print(f"       Attack  {cm[1,0]:>6,}  {cm[1,1]:>6,}")
        
        # Classification report
        print('\nClassification Report:')
        print(classification_report(y_test, y_pred, target_names=['Benign', 'Attack'], digits=4))
    
    return acc, prec, rec, f1, avrg


def save_model_pytorch(model, name, save_dir=OUT_DIR):
    """
    Save PyTorch model.
    
    Args:
        model: PyTorch model to save
        name: Model name (without extension)
        save_dir: Directory to save model
    """
    os.makedirs(save_dir, exist_ok=True)
    save_path = os.path.join(save_dir, f'{name}.pt')
    torch.save(model.state_dict(), save_path)
    print(f'✅ PyTorch model saved to: {save_path}')


def load_model_pytorch(model_class, model_args, name, save_dir=OUT_DIR):
    """
    Load PyTorch model.
    
    Args:
        model_class: Model class (e.g., MLPDropout, DeepCNN)
        model_args: Dictionary of arguments for model initialization
        name: Model name (without extension)
        save_dir: Directory where model is saved
        
    Returns:
        Loaded PyTorch model
    """
    load_path = os.path.join(save_dir, f'{name}.pt')
    model = model_class(**model_args)
    model.load_state_dict(torch.load(load_path, map_location=DEVICE))
    model = model.to(DEVICE)
    model.eval()
    print(f'✅ PyTorch model loaded from: {load_path}')
    return model


def save_Sklearn(model, name, save_dir=OUT_DIR):
    """
    Save scikit-learn model using pickle.
    
    Args:
        model: Scikit-learn model to save
        name: Model name (without extension)
        save_dir: Directory to save model
    """
    import pickle
    os.makedirs(save_dir, exist_ok=True)
    save_path = os.path.join(save_dir, f'{name}.pkl')
    with open(save_path, 'wb') as file:
        pickle.dump(model, file)
    print(f'✅ Scikit-learn model saved to: {save_path}')


def load_Sklearn(name, save_dir=OUT_DIR):
    """
    Load scikit-learn model using pickle.
    
    Args:
        name: Model name (without extension)
        save_dir: Directory where model is saved
        
    Returns:
        Loaded scikit-learn model
    """
    import pickle
    load_path = os.path.join(save_dir, f'{name}.pkl')
    with open(load_path, 'rb') as file:
        model = pickle.load(file)
    print(f'✅ Scikit-learn model loaded from: {load_path}')
    return model


# =============================================================================
# MODEL SELECTION - Comment/uncomment to choose which models to train
# =============================================================================
MODELS_TO_TRAIN = {
    # Apollon-only run (per request): train & evaluate ONLY the Apollon MAB
    'mlp_model': False,
    'cnn_model': False,
    'tcn_model': False,
    'bilstm_model': False,
    'apollon_model': True,
}

TRAIN_DEEP_MODELS = any([
    MODELS_TO_TRAIN.get('mlp_model', False),
    MODELS_TO_TRAIN.get('cnn_model', False),
    MODELS_TO_TRAIN.get('tcn_model', False),
    MODELS_TO_TRAIN.get('bilstm_model', False),
])



# =============================================================================
# APOLLON ARCHITECTURE (Multi-Armed Bandit with Thompson Sampling)
# =============================================================================

class MultiArmedBanditThompsonSampling:
    """
    Apollon: A Robust Defence System against Adversarial Machine Learning Attacks in IDS
    
    This implementation uses Multi-Armed Bandits (MAB) with Thompson Sampling to dynamically
    select the optimal classifier or ensemble of classifiers for each input. This approach
    prevents attackers from learning the IDS behaviour and generating adversarial examples.
    
    Reference: Apollon paper - "Apollon: A Robust Defence System against Adversarial 
    Machine Learning Attacks in Intrusion Detection Systems"
    
    Supports GPU acceleration via cuML (RAPIDS) when available.
    """
    
    def __init__(self, n_arms=5, n_clusters=2, seed=42):
        """
        Initialize the Multi-Armed Bandit with Thompson Sampling.
        
        Parameters:
            n_arms: Number of classifier arms (default: 5 - RF, DT, NB, LR, MLP)
            n_clusters: Number of clusters for input space partitioning
            seed: Random seed for reproducibility
        """
        self.n_arms = n_arms
        self.n_clusters = n_clusters
        self.seed = seed
        self.use_gpu = USE_CUML
        np.random.seed(seed)
        
        # Initialize classifier arms (GPU-accelerated if cuML available)
        if USE_CUML:
            self.arms = [
                RandomForestClassifier(n_estimators=100, random_state=seed),
                DecisionTreeClassifier(random_state=seed),
                GaussianNB(),
                LogisticRegression(max_iter=1000),
                skMLP(hidden_layer_sizes=(32,), max_iter=200, random_state=seed, 
                     batch_size=200, early_stopping=True, activation='tanh', solver='adam')
            ]
        else:
            self.arms = [
                RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=seed),
                DecisionTreeClassifier(random_state=seed),
                GaussianNB(),
                LogisticRegression(max_iter=1000, random_state=seed),
                skMLP(hidden_layer_sizes=(32,), max_iter=200, random_state=seed, 
                     batch_size=200, early_stopping=True, activation='tanh', solver='adam')
            ]
        self.arm_names = ['RandomForest', 'DecisionTree', 'NaiveBayes', 'LogisticRegression', 'MLP']
        
        self.cluster_centers = None
        self.cluster_assignments = None
        self.reward_sums = {}
        for cluster in range(n_clusters):
            self.reward_sums[cluster] = np.zeros(n_arms)
        
        # Beta distribution parameters for Thompson Sampling
        self.alpha = np.ones(self.n_arms)
        self.beta = np.ones(self.n_arms)
        
        self.training_time = 0.0
    
    def _to_numpy(self, arr):
        """Convert cupy/cudf arrays to numpy if needed."""
        if self.use_gpu:
            try:
                if hasattr(arr, 'get'):
                    return arr.get()
                elif hasattr(arr, 'to_numpy'):
                    return arr.to_numpy()
            except:
                pass
        return np.asarray(arr)
    
    def train(self, X_train, y_train):
        """
        Train all classifier arms and cluster the input space.
        
        Parameters:
            X_train: Training features
            y_train: Training labels
        """
        start_time = time.time()
        
        # Cluster the training data (GPU-accelerated if cuML available)
        print(f"    Clustering input space into {self.n_clusters} clusters... {'[GPU]' if self.use_gpu else '[CPU]'}")
        if self.use_gpu:
            # cuML KMeans does not support all sklearn kwargs (e.g., n_init)
            kmeans = KMeansCluster(n_clusters=self.n_clusters, random_state=self.seed)
        else:
            kmeans = KMeansCluster(n_clusters=self.n_clusters, random_state=self.seed, n_init=10)
        self.cluster_assignments = self._to_numpy(kmeans.fit_predict(X_train))
        self.cluster_centers = self._to_numpy(kmeans.cluster_centers_)
        
        # Print cluster distribution
        for i in range(self.n_clusters):
            cluster_count = np.sum(self.cluster_assignments == i)
            print(f"      Cluster {i}: {cluster_count:,} samples ({cluster_count/len(y_train)*100:.2f}%)")
        
        # Train all arms on the full training data
        print(f"\n    Training {self.n_arms} classifier arms... {'[GPU]' if self.use_gpu else '[CPU]'}")
        for arm_idx, (arm, name) in enumerate(zip(self.arms, self.arm_names)):
            print(f"      [{arm_idx+1}/{self.n_arms}] Training {name}...")
            arm.fit(X_train, y_train)
        
        # Calculate reward sums for each arm in each cluster
        print(f"\n    Computing arm rewards per cluster...")
        for cluster_idx in range(self.n_clusters):
            cluster_mask = self.cluster_assignments == cluster_idx
            cluster_X = X_train[cluster_mask]
            cluster_y = y_train[cluster_mask]
            
            for arm_idx, arm in enumerate(self.arms):
                if len(cluster_X) > 0:
                    arm_preds = self._to_numpy(arm.predict(cluster_X))
                    accuracy = np.mean(arm_preds == cluster_y)
                    self.reward_sums[cluster_idx][arm_idx] = accuracy
                    print(f"      Cluster {cluster_idx}, {self.arm_names[arm_idx]}: {accuracy*100:.2f}% accuracy")
        
        self.training_time = time.time() - start_time
        print(f"\n    ✅ Training completed in {self.training_time:.2f} seconds")
    
    def select_arm(self, cluster):
        """
        Select an arm using Thompson Sampling for the given cluster.
        
        Parameters:
            cluster: Cluster index
            
        Returns:
            Selected arm index
        """
        theta = np.zeros(self.n_arms)
        for arm_idx in range(self.n_arms):
            # Sample from Beta distribution
            alpha_param = self.alpha[arm_idx] + self.reward_sums[cluster][arm_idx]
            beta_param = self.beta[arm_idx] + 1 - self.reward_sums[cluster][arm_idx]
            theta[arm_idx] = np.random.beta(alpha_param, beta_param)
        return np.argmax(theta)
    
    def predict(self, X_test):
        """
        Predict using Thompson Sampling to dynamically select classifiers.
        
        Parameters:
            X_test: Test features
            
        Returns:
            predictions: Predicted labels
            selected_arms: Arms selected for each sample
        """
        # Assign each sample to a cluster
        clusters = np.zeros(len(X_test), dtype=int)
        for i in range(len(X_test)):
            distances = np.linalg.norm(self.cluster_centers - X_test[i], axis=1)
            clusters[i] = np.argmin(distances)
        
        # Select arm for each sample using Thompson Sampling
        selected_arms = np.zeros(len(X_test), dtype=int)
        for i in range(len(X_test)):
            selected_arms[i] = self.select_arm(clusters[i])
        
        # Predict using selected arms
        predictions = np.zeros(len(X_test), dtype=int)
        for arm_idx in range(self.n_arms):
            arm_mask = selected_arms == arm_idx
            if np.sum(arm_mask) > 0:
                arm_X = X_test[arm_mask]
                predictions[arm_mask] = self._to_numpy(self.arms[arm_idx].predict(arm_X))
        
        return predictions, selected_arms
    
    def predict_proba(self, X_test):
        """
        Predict probabilities using Thompson Sampling.
        
        Parameters:
            X_test: Test features
            
        Returns:
            probabilities: Probability of positive class (Attack)
        """
        predictions, selected_arms = self.predict(X_test)
        
        # Get probabilities from each arm
        probabilities = np.zeros(len(X_test))
        for arm_idx in range(self.n_arms):
            arm_mask = selected_arms == arm_idx
            if np.sum(arm_mask) > 0:
                arm_X = X_test[arm_mask]
                if hasattr(self.arms[arm_idx], 'predict_proba'):
                    probs = self.arms[arm_idx].predict_proba(arm_X)
                    if probs.shape[1] > 1:
                        probabilities[arm_mask] = probs[:, 1]
                    else:
                        probabilities[arm_mask] = probs[:, 0]
                else:
                    # For classifiers without predict_proba, use predictions
                    probabilities[arm_mask] = predictions[arm_mask]
        
        return probabilities


# =============================================================================
# DATA LOADING
# =============================================================================
print("\n" + "="*80)
print("DATA LOADING - CIC-DDoS2019")
print("="*80)

# Load first file - returns (flows_ok, flows_ddos)
flows_ok, flows_ddos = load_huge_file('/kaggle/input/cic-ddos2019-30gb-full-dataset-csv-files/01-12/TFTP.csv')
print('file 1 loaded')

# List of remaining files
files = [
    "DrDoS_LDAP.csv", "DrDoS_MSSQL.csv", "DrDoS_NetBIOS.csv",
    "DrDoS_NTP.csv", "DrDoS_SNMP.csv", "DrDoS_SSDP.csv",
    "DrDoS_UDP.csv", "Syn.csv", "DrDoS_DNS.csv", "UDPLag.csv"
]

# Process each file - keep benign and attack separate across all files
for i, file in enumerate(files, start=2):
    a, b = load_file(f'/kaggle/input/cic-ddos2019-30gb-full-dataset-csv-files/01-12/{file}')
    
    # Append benign flows separately
    flows_ok = pd.concat([flows_ok, a], ignore_index=True)
    # Append attack flows separately
    flows_ddos = pd.concat([flows_ddos, b], ignore_index=True)
    
    print(f'file {i} loaded')

# NOW concatenate benign and attack together (only once at the end)
samples = pd.concat([flows_ok, flows_ddos], ignore_index=True)

# Preprocess training data
samples = preprocess_dataframe(samples, dataset_type='train')

# Save to CSV
samples.to_csv('/kaggle/working/export_dataframe_proc.csv', index=False, header=True)

# Delete large variables
del flows_ok, flows_ddos, a, b

# =============================================================================
# LOAD TEST DATA (Day 2 - 03-11)
# =============================================================================
print("\n" + "-" * 80)
print("LOADING TEST DATA (Day 2)")
print("-" * 80)

# File paths
base_path = "/kaggle/input/cic-ddos2019-30gb-full-dataset-csv-files/03-11/"
files = ["LDAP.csv", "MSSQL.csv", "NetBIOS.csv", "Portmap.csv", "Syn.csv"]
# Uncomment if fixed
# files += ["UDP.csv", "UDPLag.csv"]

# Load first file - returns (flows_ok, flows_ddos)
flows_ok, flows_ddos = load_file(base_path + files[0])
print('file 1 loaded')

# Load remaining files - keep benign and attack separate
for i, file in enumerate(files[1:], start=2):
    a, b = load_file(base_path + file)
    
    # Append benign flows separately
    flows_ok = pd.concat([flows_ok, a], ignore_index=True)
    # Append attack flows separately
    flows_ddos = pd.concat([flows_ddos, b], ignore_index=True)
    
    print(f'file {i} loaded')

# NOW concatenate benign and attack together (only once at the end)
tests = pd.concat([flows_ok, flows_ddos], ignore_index=True)

# Save raw test data to CSV
tests.to_csv('/kaggle/working/export_tests.csv', index=False, header=True)

# Preprocess test data
tests = preprocess_dataframe(tests, dataset_type='test')

# Save preprocessed test data to CSV
tests.to_csv('/kaggle/working/export_tests_proc.csv', index=False, header=True)

# Free memory
del flows_ok, flows_ddos, a, b

print("="*80 + "\n")


# =============================================================================
# DATA PREPARATION FOR TRAINING
# =============================================================================
print("\n" + "="*80)
print("DATA PREPARATION")
print("="*80)

# Use Day 1 (01-12) for training (100% - no validation split)
# Use Day 2 (03-11) for testing (temporal evaluation)
print("\n⚙️ Preparing training data (Day 1 - 01-12)...")
X_train = samples.iloc[:, :-1]  # All features
y_train = samples.iloc[:, -1]   # Labels
print(f"   Training set: {len(X_train):,} samples")

# UPSAMPLE normal flows in training set to balance classes
print("\n⚙️ Upsampling benign flows in training set...")
X_combined = pd.concat([X_train, y_train], axis=1)

# Separate benign and attack flows
is_benign = X_combined[' Label'] == 0
normal = X_combined[is_benign]
ddos = X_combined[~is_benign]

print(f"   Before upsampling - Benign: {len(normal):,}, Attack: {len(ddos):,}")

# Upsample benign to match attack count
normal_upsampled = resample(
    normal,
    replace=True,
    n_samples=len(ddos),
    random_state=SEED
)

# Combine upsampled benign with attacks
upsampled = pd.concat([normal_upsampled, ddos])

# Split back into features and labels
X_train = upsampled.iloc[:, :-1]
y_train = upsampled.iloc[:, -1]

print(f"   After upsampling - Benign: {len(normal_upsampled):,}, Attack: {len(ddos):,}")
print(f"   Final training set: {len(X_train):,} samples")

# Free memory
del X_combined, normal, ddos, normal_upsampled, upsampled

# Prepare test data (Day 2)
print("\n⚙️ Preparing test data (Day 2)...")
X_test = tests.iloc[:, :-1]
y_test = tests.iloc[:, -1]
print(f"   Test set: {len(X_test):,} samples")

# Normalize data
print("\n⚙️ Normalizing features (MinMaxScaler -1 to 1)...")
X_train_np, X_test_np = normalize_data(X_train.values, X_test.values)

print(f"   Feature range: [{X_train_np.min():.2f}, {X_train_np.max():.2f}]")

# Get input dimension
input_size = X_train_np.shape[1]
print(f"\n📊 Input dimension: {input_size} features")

# Convert to PyTorch tensors
if TRAIN_DEEP_MODELS:
    print("\n⚙️ Converting to PyTorch tensors...")
    X_train_tensor = torch.FloatTensor(X_train_np)
    y_train_tensor = torch.LongTensor(y_train.values)
    X_test_tensor = torch.FloatTensor(X_test_np)
    y_test_tensor = torch.LongTensor(y_test.values)

    # Create DataLoaders
    batch_size = 256
    train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
    test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4 if not USE_XLA else 0, pin_memory=True if not USE_XLA else False)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=4 if not USE_XLA else 0, pin_memory=True if not USE_XLA else False)

    # Wrap with TPU ParallelLoader for multi-core data distribution
    if USE_XLA and xm is not None:
        import torch_xla.distributed.parallel_loader as pl
        train_loader = pl.MpDeviceLoader(train_loader, DEVICE)
        test_loader = pl.MpDeviceLoader(test_loader, DEVICE)
        print("✅ TPU ParallelLoader enabled for efficient multi-core training")

    print(f"   Batch size: {batch_size}")
    print(f"   Train batches: {len(train_loader)}")
    print(f"   Test batches: {len(test_loader)}")
else:
    print("\nℹ️ Skipping PyTorch tensor/DataLoader setup (deep models disabled; Apollon-only run)")

# Class distribution
print("\n📊 Class Distribution:")
print(f"   Training (Day 1) - Benign: {(y_train==0).sum():,}, Attack: {(y_train==1).sum():,}")
print(f"   Test (Day 2)     - Benign: {(y_test==0).sum():,}, Attack: {(y_test==1).sum():,}")

print("="*80 + "\n")


# =============================================================================
# TRAINING UTILITIES
# =============================================================================

def train_model(model, train_loader, epochs=10, learning_rate=0.001, model_name="Model"):
    """Train a PyTorch model with TPU/GPU optimization.
    
    Note: No validation set - using temporal evaluation (Day 1 train → Day 2 test).
    """
    print(f"\n{'='*80}")
    print(f"TRAINING: {model_name}")
    print(f"{'='*80}")
    print(f"Device: {DEVICE}, TPU: {USE_XLA}")
    print(f"Temporal Split: Day 1 (train) → Day 2 (test)")
    
    model = model.to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    
    train_losses = []
    train_accs = []
    
    for epoch in range(epochs):
        # Training phase
        model.train()
        train_loss = 0.0
        correct = 0
        total = 0
        
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}", disable=USE_XLA)  # Disable tqdm for TPU
        for batch_idx, (inputs, labels) in enumerate(pbar if not USE_XLA else train_loader):
            if not USE_XLA:
                inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            # TPU: Mark step for graph execution synchronization
            if USE_XLA and xm is not None:
                xm.mark_step()
            
            train_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
            
            if not USE_XLA:
                pbar.set_postfix({'loss': f'{loss.item():.4f}', 'acc': f'{100.*correct/total:.2f}%'})
            elif batch_idx % 50 == 0:  # Print every 50 batches for TPU
                print(f"  Batch {batch_idx}/{len(train_loader)} - Loss: {loss.item():.4f}, Acc: {100.*correct/total:.2f}%")
        
        train_loss /= len(train_loader)
        train_acc = correct / total
        
        train_losses.append(train_loss)
        train_accs.append(train_acc)
        
        print(f"Epoch {epoch+1}/{epochs} - Train Loss: {train_loss:.4f}, Train Acc: {train_acc*100:.2f}%")
    
    print(f"\n✅ Training complete!")
    
    return model, {'train_losses': train_losses, 'train_accs': train_accs}


def evaluate_model(model, test_loader, model_name="Model"):
    """Evaluate a PyTorch model on test set."""
    print(f"\n{'='*80}")
    print(f"EVALUATION: {model_name}")
    print(f"{'='*80}")
    
    model = model.to(DEVICE)
    model.eval()
    
    all_preds = []
    all_probs = []
    all_labels = []
    
    with torch.no_grad():
        for inputs, labels in tqdm(test_loader, desc="Evaluating", disable=USE_XLA):
            if not USE_XLA:
                inputs = inputs.to(DEVICE)
            outputs = model(inputs)
            probs = F.softmax(outputs, dim=1)[:, 1]
            _, predicted = outputs.max(1)
            
            # TPU: Mark step for evaluation
            if USE_XLA and xm is not None:
                xm.mark_step()
            
            all_preds.extend(predicted.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
            all_labels.extend(labels.cpu().numpy() if USE_XLA else labels.numpy())
    
    all_preds = np.array(all_preds)
    all_probs = np.array(all_probs)
    all_labels = np.array(all_labels)
    
    # Calculate metrics
    acc = accuracy_score(all_labels, all_preds)
    prec = precision_score(all_labels, all_preds, zero_division=0)
    rec = recall_score(all_labels, all_preds, zero_division=0)
    f1 = f1_score(all_labels, all_preds, zero_division=0)
    auc_score = roc_auc_score(all_labels, all_probs)
    
    # Detection rates
    norm_detect, atk_detect = test_normal_atk(all_labels, all_preds)
    
    # False positive rate
    tn, fp, fn, tp = confusion_matrix(all_labels, all_preds).ravel()
    fp_rate = fp / (fp + tn) if (fp + tn) > 0 else 0
    
    print(f"\n📊 Test Set Evaluation:")
    print("-" * 80)
    print(f"    Accuracy:           {acc*100:>6.2f}%")
    print(f"    Precision:          {prec*100:>6.2f}%")
    print(f"    Recall:             {rec*100:>6.2f}%")
    print(f"    F1 Score:           {f1*100:>6.2f}%")
    print(f"    AUC-ROC:            {auc_score:>8.4f}")
    print(f"    Normal Detect Rate: {norm_detect*100:>6.2f}%")
    print(f"    Attack Detect Rate: {atk_detect*100:>6.2f}%")
    print(f"    False Positive Rate:{fp_rate*100:>6.2f}%")
    print("\n    Confusion Matrix:")
    cm = confusion_matrix(all_labels, all_preds)
    print(f"                Predicted")
    print(f"                Benign  Attack")
    print(f"    Actual Benign  {cm[0,0]:>6,}  {cm[0,1]:>6,}")
    print(f"           Attack  {cm[1,0]:>6,}  {cm[1,1]:>6,}")
    print("\n    Classification Report:")
    print(classification_report(all_labels, all_preds, target_names=['Benign', 'Attack'], digits=4))
    
    return {
        'accuracy': acc,
        'precision': prec,
        'recall': rec,
        'f1': f1,
        'auc': auc_score,
        'normal_detect_rate': norm_detect,
        'attack_detect_rate': atk_detect,
        'false_positive_rate': fp_rate,
        'predictions': all_preds,
        'probabilities': all_probs,
        'labels': all_labels,
        'confusion_matrix': cm
    }


# =============================================================================
# MODEL TRAINING
# =============================================================================
print("\n" + "="*80)
print("MODEL TRAINING")
print("="*80)

trained_models = {}
results = {}

# Train MLP
if MODELS_TO_TRAIN.get('mlp_model', False):
    mlp_model = MLPDropout(d_in=input_size, d_out=2)
    mlp_model, mlp_history = train_model(mlp_model, train_loader, epochs=10, model_name="MLP")
    trained_models['mlp'] = mlp_model
    results['mlp'] = evaluate_model(mlp_model, test_loader, model_name="MLP")
    
    # Save model
    torch.save(mlp_model.state_dict(), os.path.join(OUT_DIR, 'mlp_model.pt'))

# Train CNN
if MODELS_TO_TRAIN.get('cnn_model', False):
    cnn_model = DeepCNN(d_in=input_size, d_out=2)
    cnn_model, cnn_history = train_model(cnn_model, train_loader, epochs=10, model_name="CNN")
    trained_models['cnn'] = cnn_model
    results['cnn'] = evaluate_model(cnn_model, test_loader, model_name="CNN")
    
    # Save model
    torch.save(cnn_model.state_dict(), os.path.join(OUT_DIR, 'cnn_model.pt'))

# Train TCN
if MODELS_TO_TRAIN.get('tcn_model', False):
    tcn_model = DeepTCN(d_in=input_size, d_out=2)
    tcn_model, tcn_history = train_model(tcn_model, train_loader, epochs=10, model_name="TCN")
    trained_models['tcn'] = tcn_model
    results['tcn'] = evaluate_model(tcn_model, test_loader, model_name="TCN")
    
    # Save model
    torch.save(tcn_model.state_dict(), os.path.join(OUT_DIR, 'tcn_model.pt'))

# Train BiLSTM-Attention
if MODELS_TO_TRAIN.get('bilstm_model', False):
    bilstm_model = BiLSTMAttention(d_in=input_size, n_classes=2)
    bilstm_model, bilstm_history = train_model(bilstm_model, train_loader, epochs=10, model_name="BiLSTM-Attention")
    trained_models['bilstm'] = bilstm_model
    results['bilstm'] = evaluate_model(bilstm_model, test_loader, model_name="BiLSTM-Attention")
    
    # Save model
    torch.save(bilstm_model.state_dict(), os.path.join(OUT_DIR, 'bilstm_model.pt'))

# Train Apollon (MAB with Thompson Sampling) - sklearn-based from compare.md
if MODELS_TO_TRAIN.get('apollon_model', False):
    print(f"\n{'='*80}")
    print(f"TRAINING: Apollon (Multi-Armed Bandit with Thompson Sampling)")
    print(f"{'='*80}")
    print(f"Training samples: {len(y_train):,}")
    print(f"Test samples: {len(y_test):,}")
    print(f"Number of clusters: 2")
    print(f"Number of arms (classifiers): 5 (RF, DT, NB, LR, MLP)")
    print("-" * 80)
    
    # Initialize and train Apollon
    apollon = MultiArmedBanditThompsonSampling(n_arms=5, n_clusters=2, seed=SEED)
    apollon.train(X_train_np, y_train.values)
    
    # Predict
    start_pred = time.time()
    apollon_preds, selected_arms = apollon.predict(X_test_np)
    pred_time = time.time() - start_pred
    print(f"\n    Prediction time: {pred_time:.4f} seconds")
    
    # Get probabilities
    apollon_probs = apollon.predict_proba(X_test_np)
    
    # Convert to numpy
    y_test_np_apollon = y_test.values
    
    # Calculate metrics
    apollon_acc = accuracy_score(y_test_np_apollon, apollon_preds)
    apollon_prec = precision_score(y_test_np_apollon, apollon_preds, zero_division=0)
    apollon_rec = recall_score(y_test_np_apollon, apollon_preds, zero_division=0)
    apollon_f1 = f1_score(y_test_np_apollon, apollon_preds, zero_division=0)
    apollon_auc = roc_auc_score(y_test_np_apollon, apollon_probs)
    
    # Arm selection statistics
    print(f"\n    Arm Selection Statistics:")
    for arm_idx, arm_name in enumerate(apollon.arm_names):
        count = np.sum(selected_arms == arm_idx)
        print(f"      {arm_name}: {count:,} ({count/len(y_test_np_apollon)*100:.2f}%)")
    
    # Detection rates
    norm_detect, atk_detect = test_normal_atk(y_test_np_apollon, apollon_preds)
    
    # False positive rate
    tn, fp, fn, tp = confusion_matrix(y_test_np_apollon, apollon_preds).ravel()
    fp_rate = fp / (fp + tn) if (fp + tn) > 0 else 0
    
    print(f"\n📊 Apollon Test Set Evaluation:")
    print("-" * 80)
    print(f"    Accuracy:           {apollon_acc*100:>6.2f}%")
    print(f"    Precision:          {apollon_prec*100:>6.2f}%")
    print(f"    Recall:             {apollon_rec*100:>6.2f}%")
    print(f"    F1 Score:           {apollon_f1*100:>6.2f}%")
    print(f"    AUC-ROC:            {apollon_auc:>8.4f}")
    print(f"    Normal Detect Rate: {norm_detect*100:>6.2f}%")
    print(f"    Attack Detect Rate: {atk_detect*100:>6.2f}%")
    print(f"    False Positive Rate:{fp_rate*100:>6.2f}%")
    print("\n    Confusion Matrix:")
    apollon_cm = confusion_matrix(y_test_np_apollon, apollon_preds)
    print(f"                Predicted")
    print(f"                Benign  Attack")
    print(f"    Actual Benign  {apollon_cm[0,0]:>6,}  {apollon_cm[0,1]:>6,}")
    print(f"           Attack  {apollon_cm[1,0]:>6,}  {apollon_cm[1,1]:>6,}")
    print("\n    Classification Report:")
    print(classification_report(y_test_np_apollon, apollon_preds, target_names=['Benign', 'Attack'], digits=4))
    
    # Store results
    results['apollon'] = {
        'accuracy': apollon_acc,
        'precision': apollon_prec,
        'recall': apollon_rec,
        'f1': apollon_f1,
        'auc': apollon_auc,
        'normal_detect_rate': norm_detect,
        'attack_detect_rate': atk_detect,
        'false_positive_rate': fp_rate,
        'predictions': apollon_preds,
        'probabilities': apollon_probs,
        'labels': y_test_np_apollon,
        'confusion_matrix': apollon_cm,
        'selected_arms': selected_arms
    }
    
    # Save Apollon model
    joblib.dump(apollon, os.path.join(OUT_DIR, 'apollon_model.pkl'))
    print(f"\n💾 Apollon model saved to: {os.path.join(OUT_DIR, 'apollon_model.pkl')}")

# Find best model
best_f1 = 0.0
best_model_name = None
for name, result in results.items():
    if result['f1'] > best_f1:
        best_f1 = result['f1']
        best_model_name = name

print(f"\n🏆 Best model: {best_model_name.upper()} (F1: {best_f1*100:.2f}%)")


# =============================================================================
# PLOTTING FUNCTIONS
# =============================================================================

def plot_confusion_matrix(cm, model_name, save_path):
    """Plot confusion matrix heatmap."""
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=['Benign', 'Attack'], 
                yticklabels=['Benign', 'Attack'])
    plt.title(f'Confusion Matrix - {model_name}')
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.close()


def plot_roc_curve(labels, probs, model_name, save_path):
    """Plot ROC curve."""
    fpr, tpr, _ = roc_curve(labels, probs)
    roc_auc = auc(fpr, tpr)
    
    plt.figure(figsize=(8, 6))
    plt.plot(fpr, tpr, label=f'{model_name} (AUC = {roc_auc:.4f})', linewidth=2)
    plt.plot([0, 1], [0, 1], 'k--', label='Random Classifier')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(f'ROC Curve - {model_name}')
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.close()


def plot_pr_curve(labels, probs, model_name, save_path):
    """Plot Precision-Recall curve."""
    precision, recall, _ = precision_recall_curve(labels, probs)
    pr_auc = auc(recall, precision)
    
    plt.figure(figsize=(8, 6))
    plt.plot(recall, precision, label=f'{model_name} (AUC = {pr_auc:.4f})', linewidth=2)
    plt.xlabel('Recall')
    plt.ylabel('Precision')
    plt.title(f'Precision-Recall Curve - {model_name}')
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.close()


def plot_metrics_comparison(results, save_path):
    """Plot bar chart comparing model metrics."""
    metrics = ['accuracy', 'precision', 'recall', 'f1', 'auc']
    model_names = list(results.keys())
    
    fig, ax = plt.subplots(figsize=(12, 6))
    x = np.arange(len(metrics))
    width = 0.8 / len(model_names)
    
    for i, model_name in enumerate(model_names):
        values = [results[model_name][m] * 100 for m in metrics]
        ax.bar(x + i * width, values, width, label=model_name.upper())
    
    ax.set_xlabel('Metrics')
    ax.set_ylabel('Score (%)')
    ax.set_title('Model Performance Comparison')
    ax.set_xticks(x + width * (len(model_names) - 1) / 2)
    ax.set_xticklabels([m.upper() for m in metrics])
    ax.legend()
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.close()


# Generate plots for each model
print("\n" + "="*80)
print("GENERATING PLOTS")
print("="*80)

for name, result in results.items():
    print(f"\nGenerating plots for {name.upper()}...")
    
    plot_confusion_matrix(result['confusion_matrix'], name.upper(), 
                         os.path.join(PLOT_DIR, f'{name}_confusion_matrix.png'))
    plot_roc_curve(result['labels'], result['probabilities'], name.upper(), 
                  os.path.join(PLOT_DIR, f'{name}_roc_curve.png'))
    plot_pr_curve(result['labels'], result['probabilities'], name.upper(), 
                 os.path.join(PLOT_DIR, f'{name}_pr_curve.png'))

# Comparison plot
if len(results) > 1:
    plot_metrics_comparison(results, os.path.join(PLOT_DIR, 'model_comparison.png'))

print("\n✅ All plots saved to:", PLOT_DIR)
print("="*80 + "\n")


# =============================================================================
# ENSEMBLE EVALUATION (Baseline - Static Mean Logits)
# =============================================================================
if len(trained_models) > 1:
    print("\n" + "="*80)
    print("ENSEMBLE EVALUATION (BASELINE)")
    print("="*80)
    print("Strategy: Static Mean Logits (all models weighted equally)")
    print(f"Models in ensemble: {list(trained_models.keys())}")
    print("-" * 80)
    
    # Evaluate ensemble
    all_preds = []
    all_probs = []
    all_labels = []
    
    for model in trained_models.values():
        model.eval()
    
    with torch.no_grad():
        for inputs, labels in test_loader:
            if not USE_XLA:
                inputs = inputs.to(DEVICE)
            
            # Collect logits from all models
            logits_list = []
            for model in trained_models.values():
                logits = model(inputs)
                logits_list.append(logits)
            
            # Mean logits ensemble
            ensemble_logits = torch.stack(logits_list, dim=0).mean(dim=0)
            probs = F.softmax(ensemble_logits, dim=1)[:, 1]
            _, predicted = torch.max(ensemble_logits.data, 1)
            
            all_preds.extend(predicted.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
            all_labels.extend(labels.cpu().numpy() if USE_XLA else labels.numpy())
    
    # Calculate metrics
    all_preds = np.array(all_preds)
    all_probs = np.array(all_probs)
    all_labels = np.array(all_labels)
    
    ensemble_accuracy = accuracy_score(all_labels, all_preds)
    ensemble_precision = precision_score(all_labels, all_preds, zero_division=0)
    ensemble_recall = recall_score(all_labels, all_preds, zero_division=0)
    ensemble_f1 = f1_score(all_labels, all_preds, zero_division=0)
    ensemble_auc = roc_auc_score(all_labels, all_probs)
    
    print(f"\n📊 Ensemble Test Set Evaluation:")
    print("-" * 80)
    print(f"    Accuracy:  {ensemble_accuracy*100:>6.2f}%")
    print(f"    Precision: {ensemble_precision*100:>6.2f}%")
    print(f"    Recall:    {ensemble_recall*100:>6.2f}%")
    print(f"    F1 Score:  {ensemble_f1*100:>6.2f}%")
    print(f"    AUC-ROC:   {ensemble_auc:>8.4f}")
    print("\n    Confusion Matrix:")
    ensemble_cm = confusion_matrix(all_labels, all_preds)
    print(f"                Predicted")
    print(f"                Benign  Attack")
    print(f"    Actual Benign  {ensemble_cm[0,0]:>6,}  {ensemble_cm[0,1]:>6,}")
    print(f"           Attack  {ensemble_cm[1,0]:>6,}  {ensemble_cm[1,1]:>6,}")
    print("\n    Classification Report:")
    print(classification_report(all_labels, all_preds, 
                                target_names=['Benign', 'Attack'], digits=4))
    
    # Add to results for comparison
    results['ensemble'] = {
        'accuracy': ensemble_accuracy,
        'precision': ensemble_precision,
        'recall': ensemble_recall,
        'f1': ensemble_f1,
        'auc': ensemble_auc,
        'predictions': all_preds,
        'probabilities': all_probs,
        'labels': all_labels,
        'confusion_matrix': ensemble_cm
    }
    
    # Generate ensemble plots
    plot_confusion_matrix(ensemble_cm, 'Ensemble (Mean Logits)', 
                         os.path.join(PLOT_DIR, 'ensemble_confusion_matrix.png'))
    plot_roc_curve(all_labels, all_probs, 'Ensemble (Mean Logits)', 
                  os.path.join(PLOT_DIR, 'ensemble_roc_curve.png'))
    plot_pr_curve(all_labels, all_probs, 'Ensemble (Mean Logits)', 
                 os.path.join(PLOT_DIR, 'ensemble_pr_curve.png'))
    
    # Update comparison plots with ensemble
    plot_metrics_comparison(results, os.path.join(PLOT_DIR, 'model_comparison_with_ensemble.png'))
    
    # Print improvement over best single model
    if best_f1 < ensemble_f1:
        improvement = (ensemble_f1 - best_f1) * 100
        print(f"\n🎯 Ensemble improvement: +{improvement:.2f}% F1 over best single model")
    else:
        degradation = (best_f1 - ensemble_f1) * 100
        print(f"\n⚠️ Ensemble degradation: -{degradation:.2f}% F1 vs best single model")

print("\n" + "="*80)
print("TRAINING COMPLETE")
print("="*80 + "\n")

# =============================================================================
# FINAL RESULTS TABLE
# =============================================================================
print("\n" + "="*80)
print("FINAL RESULTS SUMMARY")
print("="*80)
print(f"\n{'Model':<15} {'Accuracy':<10} {'Precision':<11} {'Recall':<10} {'F1 Score':<10} {'AUC-ROC':<10} {'Normal Det':<11} {'Attack Det':<11} {'False Pos':<10}")
print("-" * 80)

for model_name, result in results.items():
    norm_det, atk_det = test_normal_atk(result['labels'], result['predictions'])
    tn, fp, fn, tp = confusion_matrix(result['labels'], result['predictions']).ravel()
    fp_rate = fp / (fp + tn) if (fp + tn) > 0 else 0
    
    print(f"{model_name.upper():<15} {result['accuracy']*100:>8.2f}%  {result['precision']*100:>9.2f}%  {result['recall']*100:>8.2f}%  {result['f1']*100:>8.2f}%  {result['auc']:>8.4f}  {norm_det*100:>9.2f}%  {atk_det*100:>9.2f}%  {fp_rate*100:>8.2f}%")

print("="*80 + "\n")






/usr/local/lib/python3.12/dist-packages/sqlalchemy/orm/query.py:195: SyntaxWarning: "is not" with 'tuple' literal. Did you mean "!="?
  if entities is not ():



⚠️ cuML not available - Apollon will use CPU sklearn
🔧 Apollon using CPU sklearn classifiers

HARDWARE DETECTION
PyTorch: 2.8.0+cu126
Platform: Linux-6.6.105+-x86_64-with-glibc2.35
⚠️ TPU not available (torch_xla not installed)
   For Kaggle TPU: Runtime → Change runtime type → TPU

✅ CUDA Available
   GPU Count: 2
   GPU 0: Tesla T4 (15.83 GB)
   GPU 1: Tesla T4 (15.83 GB)

🎯 Using CUDA: cuda


DATA LOADING - CIC-DDoS2019
file 1 loaded
file 2 loaded
file 3 loaded
file 4 loaded
file 5 loaded
file 6 loaded
file 7 loaded
file 8 loaded
file 9 loaded
file 10 loaded
file 11 loaded


/tmp/ipykernel_24/744037612.py:359: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[' Label'] = df[' Label'].replace(label_map).astype(int)



--------------------------------------------------------------------------------
LOADING TEST DATA (Day 2)
--------------------------------------------------------------------------------
file 1 loaded
file 2 loaded
file 3 loaded
file 4 loaded
file 5 loaded


/tmp/ipykernel_24/744037612.py:359: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[' Label'] = df[' Label'].replace(label_map).astype(int)




DATA PREPARATION

⚙️ Preparing training data (Day 1 - 01-12)...
   Training set: 331,879 samples

⚙️ Upsampling benign flows in training set...
   Before upsampling - Benign: 56,863, Attack: 275,016
   After upsampling - Benign: 275,016, Attack: 275,016
   Final training set: 550,032 samples

⚙️ Preparing test data (Day 2)...
   Test set: 298,578 samples

⚙️ Normalizing features (MinMaxScaler -1 to 1)...
   Feature range: [-1.00, 1.00]

📊 Input dimension: 82 features

ℹ️ Skipping PyTorch tensor/DataLoader setup (deep models disabled; Apollon-only run)

📊 Class Distribution:
   Training (Day 1) - Benign: 275,016, Attack: 275,016
   Test (Day 2)     - Benign: 49,763, Attack: 248,815


MODEL TRAINING

TRAINING: Apollon (Multi-Armed Bandit with Thompson Sampling)
Training samples: 550,032
Test samples: 298,578
Number of clusters: 2
Number of arms (classifiers): 5 (RF, DT, NB, LR, MLP)
--------------------------------------------------------------------------------
    Clustering input sp